# Snake Classifier - FIXED Pipeline (Colab Version)

**Self-contained notebook - no external imports needed**

## Setup Instructions:
1. Upload `venomous_data.zip` to your Google Drive
2. Run all cells in order

**Key Fixes Applied:**
- Dataset filtered to classes with >= 30 real images
- Split FIRST, then augment ONLY training data
- NO PCA - full feature vectors
- Controlled augmentation
- Fixed LBP parameters

## 1. Mount Google Drive & Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install -q scikit-image lightgbm

In [ ]:
# Unzip dataset (adjust path if needed)
import os
import zipfile

# Change this path to where you uploaded the zip file
ZIP_PATH = '/content/drive/MyDrive/venomous_data.zip'
EXTRACT_PATH = '/content/venomous_data'

if os.path.exists(ZIP_PATH):
    print(f'Extracting {ZIP_PATH}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print('Done!')
else:
    print(f'ZIP file not found at {ZIP_PATH}')
    print('Please upload venomous_data.zip to your Google Drive root folder')

# Verify
if os.path.exists(EXTRACT_PATH):
    print(f'\nFound {len(os.listdir(EXTRACT_PATH))} folders in venomous_data')
else:
    # Maybe it extracted with nested folder
    if os.path.exists('/content/venomous_data/venomous_data'):
        EXTRACT_PATH = '/content/venomous_data/venomous_data'
        print(f'Found nested folder, using: {EXTRACT_PATH}')

## 2. Imports

In [ ]:
import os, sys, random, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from IPython.display import display

from skimage.feature import hog, local_binary_pattern

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, top_k_accuracy_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_fscore_support,
    f1_score
)

import lightgbm as lgb

plt.rcParams.update({'figure.dpi': 100})
PALETTE = sns.color_palette('deep')

print('Imports OK')

## 3. Configuration

In [ ]:
# ============== CONFIGURATION ==============
DATASET_PATH = EXTRACT_PATH  # Set above

# Image settings
IMG_SIZE = (256, 256)

# HOG parameters
HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)

# LBP parameters (FIXED: reduced radius)
LBP_POINTS = 16
LBP_RADIUS = 2

# HSV histogram bins
HSV_BINS = (8, 8, 8)

# Dataset filtering
MIN_REAL_IMAGES = 30
TARGET_NUM_CLASSES = 8

# Training settings
TEST_SPLIT = 0.2
RANDOM_SEED = 42
MAX_AUGMENT_RATIO = 0.5

# Cross-validation
RUN_CV = True
CV_FOLDS = 5

# Venomous species keywords
VENOMOUS_KEYWORDS = [
    'naja', 'cobra', 'bungarus', 'krait', 'viper', 'craspedocephalus',
    'hypnale', 'ophiophagus', 'daboia', 'echis', 'trimeresurus'
]

print(f'Dataset: {DATASET_PATH}')
print(f'Image size: {IMG_SIZE}')
print(f'LBP: points={LBP_POINTS}, radius={LBP_RADIUS}')

## 4. Feature Extraction Functions

In [ ]:
def is_venomous(species_name):
    """Check if species is venomous based on keywords."""
    name_lower = species_name.lower()
    for keyword in VENOMOUS_KEYWORDS:
        if keyword in name_lower:
            return True, keyword
    return False, None


def extract_hog(img_gray):
    """Extract HOG features."""
    features = hog(
        img_gray,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm='L2-Hys',
        visualize=False
    )
    return features.astype(np.float32)


def extract_lbp(img_gray):
    """Extract LBP histogram."""
    lbp = local_binary_pattern(img_gray, LBP_POINTS, LBP_RADIUS, method='uniform')
    n_bins = int(lbp.max() + 1)
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins), density=True)
    return hist.astype(np.float32)


def extract_hsv_histogram(img_bgr):
    """Extract HSV color histogram."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, HSV_BINS, [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten().astype(np.float32)


def extract_features(img_bgr):
    """Extract all features from BGR image."""
    img_bgr = cv2.resize(img_bgr, IMG_SIZE)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    
    hog_feat = extract_hog(img_gray)
    lbp_feat = extract_lbp(img_gray)
    hsv_feat = extract_hsv_histogram(img_bgr)
    
    return np.hstack([hog_feat, lbp_feat, hsv_feat]).astype(np.float32)


def extract_features_from_path(image_path):
    """Extract features from image file path."""
    try:
        img = cv2.imread(image_path)
        if img is None:
            return None
        return extract_features(img)
    except Exception as e:
        print(f'Error: {image_path}: {e}')
        return None


print('Feature extraction functions defined')

## 5. Augmentation Functions

In [ ]:
def augment_image(img, augment_type='random'):
    """Apply controlled augmentation."""
    if augment_type == 'random':
        augment_type = random.choice(['flip', 'rotate', 'brightness', 'zoom'])
    
    h, w = img.shape[:2]
    
    if augment_type == 'flip':
        img = cv2.flip(img, 1)  # Horizontal flip
    
    elif augment_type == 'rotate':
        angle = random.uniform(-15, 15)
        M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    
    elif augment_type == 'brightness':
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * random.uniform(0.8, 1.2), 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    
    elif augment_type == 'zoom':
        scale = random.uniform(0.95, 1.05)
        new_w, new_h = int(w * scale), int(h * scale)
        if scale > 1.0:
            resized = cv2.resize(img, (new_w, new_h))
            start_x, start_y = (new_w - w) // 2, (new_h - h) // 2
            img = resized[start_y:start_y + h, start_x:start_x + w]
        else:
            resized = cv2.resize(img, (new_w, new_h))
            pad_x, pad_y = (w - new_w) // 2, (h - new_h) // 2
            img = cv2.copyMakeBorder(resized, pad_y, h - new_h - pad_y,
                                      pad_x, w - new_w - pad_x, cv2.BORDER_REFLECT)
    
    return img


def generate_augmented_images(img, num_augments=2):
    """Generate multiple augmented versions."""
    aug_types = ['flip', 'rotate', 'brightness', 'zoom']
    return [augment_image(img.copy(), aug_types[i % len(aug_types)]) 
            for i in range(num_augments)]


print('Augmentation functions defined')

## 6. Dataset Analysis

In [ ]:
def analyze_dataset(dataset_path):
    """Count real vs augmented images per class."""
    analysis = []
    
    for species in sorted(os.listdir(dataset_path)):
        species_dir = os.path.join(dataset_path, species)
        if not os.path.isdir(species_dir):
            continue
        
        all_images = [f for f in os.listdir(species_dir) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        real_images = [f for f in all_images if not f.startswith('aug_')]
        
        analysis.append({
            'species': species,
            'real_count': len(real_images),
            'aug_count': len(all_images) - len(real_images),
            'total_count': len(all_images)
        })
    
    return pd.DataFrame(analysis)


# Analyze
df_analysis = analyze_dataset(DATASET_PATH)
print('Dataset Analysis:')
display(df_analysis.sort_values('real_count', ascending=False))

In [ ]:
# Select classes with enough real images
df_filtered = df_analysis[df_analysis['real_count'] >= MIN_REAL_IMAGES].copy()
df_filtered = df_filtered.sort_values('real_count', ascending=False)

if len(df_filtered) > TARGET_NUM_CLASSES:
    df_selected = df_filtered.head(TARGET_NUM_CLASSES)
else:
    df_selected = df_filtered

SELECTED_CLASSES = df_selected['species'].tolist()

print(f'\nSelected {len(SELECTED_CLASSES)} classes:')
for _, row in df_selected.iterrows():
    v, _ = is_venomous(row['species'])
    tag = 'VENOMOUS' if v else 'non-venomous'
    print(f"  {row['species']:<40} {row['real_count']:>3} real  [{tag}]")

## 7. Load Real Images Only

In [ ]:
def load_real_image_paths(dataset_path, selected_classes):
    """Load paths to REAL images only."""
    image_data = []
    
    for species in selected_classes:
        species_dir = os.path.join(dataset_path, species)
        if not os.path.isdir(species_dir):
            continue
        
        for img_file in os.listdir(species_dir):
            if img_file.startswith('aug_'):  # Skip augmented
                continue
            if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            
            img_path = os.path.join(species_dir, img_file)
            image_data.append((img_path, species))
    
    return image_data


# Load
image_data = load_real_image_paths(DATASET_PATH, SELECTED_CLASSES)
print(f'Loaded {len(image_data)} real images')

# Create labels
label_names = np.array(SELECTED_CLASSES)
label_to_idx = {name: idx for idx, name in enumerate(label_names)}

image_paths = np.array([x[0] for x in image_data])
labels = np.array([label_to_idx[x[1]] for x in image_data])

print('\nClass distribution:')
for idx, name in enumerate(label_names):
    print(f'  {name}: {(labels == idx).sum()}')

## 8. Train/Test Split (BEFORE augmentation)

In [ ]:
# CRITICAL: Split FIRST
paths_train, paths_test, y_train_raw, y_test = train_test_split(
    image_paths, labels,
    test_size=TEST_SPLIT,
    stratify=labels,
    random_state=RANDOM_SEED
)

print(f'Train: {len(paths_train)} images')
print(f'Test:  {len(paths_test)} images')

## 9. Extract Test Features (No Augmentation)

In [ ]:
print('Extracting TEST features (no augmentation)...')
t0 = time.time()

X_test_list, y_test_valid = [], []

for i, img_path in enumerate(tqdm(paths_test, desc='Test')):
    feat = extract_features_from_path(img_path)
    if feat is not None:
        X_test_list.append(feat)
        y_test_valid.append(y_test[i])

X_test = np.array(X_test_list, dtype=np.float32)
y_test = np.array(y_test_valid)

print(f'\nDone in {time.time()-t0:.1f}s')
print(f'X_test: {X_test.shape}')

## 10. Extract Train Features (With Augmentation)

In [ ]:
print('Extracting TRAIN features (with augmentation)...')
t0 = time.time()

X_train_list, y_train_list = [], []

for i, img_path in enumerate(tqdm(paths_train, desc='Train')):
    label = y_train_raw[i]
    
    # Original image
    img = cv2.imread(img_path)
    if img is None:
        continue
    
    img = cv2.resize(img, IMG_SIZE)
    feat = extract_features(img)
    X_train_list.append(feat)
    y_train_list.append(label)
    
    # Add 1 augmentation per image
    aug_img = augment_image(img.copy(), 'random')
    aug_feat = extract_features(aug_img)
    X_train_list.append(aug_feat)
    y_train_list.append(label)

X_train = np.array(X_train_list, dtype=np.float32)
y_train = np.array(y_train_list)

print(f'\nDone in {time.time()-t0:.1f}s')
print(f'X_train: {X_train.shape}')
print(f'Original: {len(paths_train)}, After augmentation: {len(X_train)}')

## 11. Feature Scaling

In [ ]:
print('Scaling features...')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# NO PCA
X_train_t = X_train_scaled
X_test_t = X_test_scaled

print(f'Feature dimension: {X_train_t.shape[1]} (NO PCA)')

## 12. Cross-Validation

In [ ]:
cv_results = {}

if RUN_CV:
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    
    # LinearSVC
    print('CV: LinearSVC...')
    svc = LinearSVC(C=1.0, max_iter=10000, class_weight='balanced', dual=False, random_state=RANDOM_SEED)
    scores = cross_val_score(svc, X_train_t, y_train, cv=skf, scoring='accuracy')
    cv_results['linearsvc'] = scores
    print(f'  Mean: {scores.mean()*100:.2f}% +/- {scores.std()*100:.2f}%')
    
    # LogReg
    print('CV: LogReg...')
    lr = LogisticRegression(max_iter=2000, solver='saga', class_weight='balanced', random_state=RANDOM_SEED)
    scores = cross_val_score(lr, X_train_t, y_train, cv=skf, scoring='accuracy')
    cv_results['logreg'] = scores
    print(f'  Mean: {scores.mean()*100:.2f}% +/- {scores.std()*100:.2f}%')
    
    # LightGBM
    print('CV: LightGBM...')
    lgbm = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, class_weight='balanced', 
                               random_state=RANDOM_SEED, verbose=-1)
    scores = cross_val_score(lgbm, X_train_t, y_train, cv=skf, scoring='accuracy')
    cv_results['lgbm'] = scores
    print(f'  Mean: {scores.mean()*100:.2f}% +/- {scores.std()*100:.2f}%')

## 13. Train Models

In [ ]:
TRAIN_MODELS = {}
train_times = {}

# LinearSVC
print('Training LinearSVC...')
t0 = time.time()
m = LinearSVC(C=1.0, max_iter=15000, class_weight='balanced', dual=False, random_state=RANDOM_SEED)
m.fit(X_train_t, y_train)
TRAIN_MODELS['linearsvc'] = m
train_times['linearsvc'] = time.time() - t0
print(f'  Done in {train_times["linearsvc"]:.1f}s')

# LogReg
print('Training LogReg...')
t0 = time.time()
m = LogisticRegression(max_iter=3000, solver='saga', class_weight='balanced', random_state=RANDOM_SEED)
m.fit(X_train_t, y_train)
TRAIN_MODELS['logreg'] = m
train_times['logreg'] = time.time() - t0
print(f'  Done in {train_times["logreg"]:.1f}s')

# LightGBM
print('Training LightGBM...')
t0 = time.time()
m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, max_depth=7,
                        class_weight='balanced', random_state=RANDOM_SEED, verbose=-1)
m.fit(X_train_t, y_train)
TRAIN_MODELS['lgbm'] = m
train_times['lgbm'] = time.time() - t0
print(f'  Done in {train_times["lgbm"]:.1f}s')

## 14. Evaluate Models

In [ ]:
eval_results = {}

for name, m in TRAIN_MODELS.items():
    y_pred = m.predict(X_test_t)
    
    if hasattr(m, 'decision_function'):
        raw_scores = m.decision_function(X_test_t)
    else:
        raw_scores = m.predict_proba(X_test_t)
    
    top1 = accuracy_score(y_test, y_pred)
    top3 = top_k_accuracy_score(y_test, raw_scores, k=3) if len(label_names) >= 3 else top1
    top5 = top_k_accuracy_score(y_test, raw_scores, k=5) if len(label_names) >= 5 else top3
    f1_macro = f1_score(y_test, y_pred, average='macro')
    
    eval_results[name] = {
        'top1': top1, 'top3': top3, 'top5': top5,
        'f1_macro': f1_macro, 'y_pred': y_pred, 'raw_scores': raw_scores
    }
    
    print(f'{name:12s}: Top-1={top1*100:.2f}%  Top-3={top3*100:.2f}%  Top-5={top5*100:.2f}%  F1={f1_macro*100:.2f}%')

## 15. Results Visualization

In [ ]:
# Accuracy comparison
models_list = list(eval_results.keys())
top1_vals = [eval_results[m]['top1'] * 100 for m in models_list]
top3_vals = [eval_results[m]['top3'] * 100 for m in models_list]
top5_vals = [eval_results[m]['top5'] * 100 for m in models_list]

x = np.arange(len(models_list))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width, top1_vals, width, label='Top-1', color=PALETTE[0])
b2 = ax.bar(x, top3_vals, width, label='Top-3', color=PALETTE[1])
b3 = ax.bar(x + width, top5_vals, width, label='Top-5', color=PALETTE[2])

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in models_list], fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_ylim(0, 110)
ax.set_title('Model Comparison - Top-K Accuracy (FIXED Pipeline)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
BEST_MODEL = max(eval_results, key=lambda n: eval_results[n]['top1'])

rows = []
for name in models_list:
    row = {
        'Model': name.upper(),
        'Top-1 (%)': round(eval_results[name]['top1'] * 100, 2),
        'Top-3 (%)': round(eval_results[name]['top3'] * 100, 2),
        'Top-5 (%)': round(eval_results[name]['top5'] * 100, 2),
        'F1 Macro (%)': round(eval_results[name]['f1_macro'] * 100, 2),
        'Time (s)': round(train_times.get(name, 0), 1),
    }
    if name in cv_results:
        row['CV Mean (%)'] = round(cv_results[name].mean() * 100, 2)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Model')
display(summary_df)

print(f'\nBest model: {BEST_MODEL.upper()} ({eval_results[BEST_MODEL]["top1"]*100:.2f}%)')

## 16. Confusion Matrix

In [ ]:
y_pred = eval_results[BEST_MODEL]['y_pred']

cm = confusion_matrix(y_test, y_pred)
cm_labels = [s.replace('_', '\n') for s in label_names]

fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay(cm, display_labels=cm_labels).plot(
    ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45, values_format='d'
)
ax.set_title(f'Confusion Matrix - {BEST_MODEL.upper()}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 17. Classification Report

In [ ]:
print('Classification Report:')
print('=' * 70)
print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

## 18. Final Summary

In [ ]:
print('=' * 70)
print('FIXED PIPELINE - FINAL SUMMARY')
print('=' * 70)
print(f'Best Model:          {BEST_MODEL.upper()}')
print(f'Number of Classes:   {len(label_names)}')
print(f'Training Samples:    {len(y_train)}')
print(f'Test Samples:        {len(y_test)}')
print(f'Feature Dimension:   {X_train_t.shape[1]}')
print(f'PCA Used:            NO')
print()
print(f'Top-1 Accuracy:      {eval_results[BEST_MODEL]["top1"]*100:.2f}%')
print(f'Top-3 Accuracy:      {eval_results[BEST_MODEL]["top3"]*100:.2f}%')
print(f'Top-5 Accuracy:      {eval_results[BEST_MODEL]["top5"]*100:.2f}%')
print(f'Macro F1 Score:      {eval_results[BEST_MODEL]["f1_macro"]*100:.2f}%')

print('\n' + '=' * 70)
print('FIXES APPLIED:')
print('=' * 70)
print('1. Dataset filtered to classes with >= 30 real images')
print('2. Only REAL images used (aug_* files excluded)')
print('3. Train/test split BEFORE any augmentation')
print('4. Augmentation applied ONLY to training data')
print('5. PCA REMOVED - full feature vectors used')
print('6. LBP radius reduced from 3 to 2')
print('7. Controlled augmentation (1 aug per image)')